In [6]:
import numpy as np
import pandas as pd
import re

data = pd.read_csv("spelling_corpus.csv")

print(data.head())
print(data.shape)

# Build vocabulary
vocab = set(data["correct_word"].str.lower())

print("Vocabulary Size:", len(vocab))

# Edit Distance
def edit_distance(s1, s2):

    m = len(s1)
    n = len(s2)

    dp = [[0 for _ in range(n + 1)] for _ in range(m + 1)]

    for i in range(m + 1):
        dp[i][0] = i

    for j in range(n + 1):
        dp[0][j] = j

    for i in range(1, m + 1):

        for j in range(1, n + 1):

            if s1[i - 1] == s2[j - 1]:
                dp[i][j] = dp[i - 1][j - 1]

            else:
                dp[i][j] = 1 + min(
                    dp[i - 1][j],
                    dp[i][j - 1],
                    dp[i - 1][j - 1]
                )

    return dp[m][n]

# Correct Word
def correct_word(word):

    word = word.lower()

    # Already correct
    if word in vocab:
        return word

    # Check spelling-error corpus first
    matches = data[
        data["misspelled_word"].str.lower() == word
    ]

    if not matches.empty:
        return matches.iloc[0]["correct_word"].lower()

    # If not found, use edit distance
    best_word = word
    min_distance = float("inf")

    for k in vocab:

        distance = edit_distance(word, k)

        if distance < min_distance:
            min_distance = distance
            best_word = k

    return best_word
    
# Correct Query
def correct_query(query):

    words = query.lower().split()

    corrected_words = []
    incorrect_words = []

    for word in words:

        if word in vocab:

            corrected_words.append(word)

        else:

            incorrect_words.append(word)

            corrected = correct_word(word)

            corrected_words.append(corrected)

    return incorrect_words, corrected_words

# User Input
query = input("\nEnter your search query: ")

incorrect, corrected = correct_query(query)

print("\nOriginal Query:")
print(query)

print("\nIncorrect Words:")
print(incorrect)

print("\nSuggested Corrections:")

for word in incorrect:
    print(word, "->", correct_word(word))

print("\nFinal Corrected Query:")
print(" ".join(corrected))

#Workflow
#spelling error corpus --> load csv dataset --> build vocab --> user query --> tokenize query --> check each word in vocab
#if word present in vocab keep word
# if word not present --> check spelling corpus --> misspelled word found --> yes --> get correct word ---> this goes to build corrected query
# if didn't found correct word --> calc edit distance --> find closest word --> correct word --> this goes to build corrected query
#build corrected query -- display og query --> display incorrect words --> display corrections --> display final query

  correct_word misspelled_word
0       Albert              Ab
1      America         Ameraca
2      America         Amercia
3     American        Ameracan
4        April           Apirl
(36133, 2)
Vocabulary Size: 6130



Enter your search query:  machne lerning cours



Original Query:
machne lerning cours

Incorrect Words:
['machne', 'lerning', 'cours']

Suggested Corrections:
machne -> mache
lerning -> learning
cours -> coarse

Final Corrected Query:
mache learning coarse
